# Flask Fundamentals - Complete Guide

Welcome to this comprehensive Flask tutorial! Flask is a lightweight and flexible Python web framework that makes it easy to build web applications. This notebook covers all the essential concepts you need to get started.

## What You'll Learn
- Setting up Flask applications
- Creating routes and handling HTTP methods
- Working with templates and static files
- Handling requests and responses
- Managing errors
- Best practices for Flask development

## 1. Import and Setup Flask Application

First, you need to install Flask and then import it to create an application instance.

In [ ]:
# First, install Flask if you haven't already
# You can uncomment and run this in terminal: pip install flask

# Import Flask
from flask import Flask, render_template, request, jsonify, url_for, redirect, session

# Create a Flask application instance
# __name__ helps Flask locate resources relative to the module
app = Flask(__name__)

# Configure the app (optional but recommended)
app.config['DEBUG'] = True  # Enable debug mode for development
app.secret_key = 'your-secret-key-change-this'  # Required for session management

print(f"Flask app created: {app}")
print(f"App name: {app.name}")
print(f"Debug mode: {app.debug}")

## 2. Create Your First Route

Routes are URL patterns that map to view functions. Use the `@app.route()` decorator to define routes.

In [ ]:
# Example 1: Basic route that returns plain text
@app.route('/')
def hello():
    """The home page route"""
    return 'Hello, World!'

# Example 2: Route that returns HTML
@app.route('/greet')
def greet():
    """A route that returns HTML"""
    return '<h1>Welcome to Flask!</h1><p>This is your first Flask application.</p>'

# Example 3: Multiple routes pointing to the same function
@app.route('/home')
@app.route('/index')
def home():
    """Multiple routes can call the same function"""
    return 'This is the home page'

# Print all registered routes
print("\nRegistered routes:")
for rule in app.url_map.iter_rules():
    print(f"  {rule.rule} -> {rule.endpoint} (methods: {rule.methods})")

## 3. Handle HTTP Methods (GET and POST)

Different HTTP methods are used for different purposes. By default, routes handle GET requests only.

In [ ]:
# Example 1: Handle only GET requests (default)
@app.route('/get-only', methods=['GET'])
def get_only():
    """This route only accepts GET requests"""
    return 'This is a GET request'

# Example 2: Handle only POST requests
@app.route('/post-only', methods=['POST'])
def post_only():
    """This route only accepts POST requests"""
    return 'This is a POST request'

# Example 3: Handle multiple methods
@app.route('/form', methods=['GET', 'POST'])
def handle_form():
    """Handle both GET and POST requests"""
    if request.method == 'POST':
        # Access form data from POST request
        name = request.form.get('name', 'Guest')
        email = request.form.get('email', 'unknown@example.com')
        return f'<h1>Received: {name} - {email}</h1>'
    else:
        # Show form for GET request
        return '''
        <form method="post">
            <input type="text" name="name" placeholder="Enter your name"><br>
            <input type="email" name="email" placeholder="Enter your email"><br>
            <button type="submit">Submit</button>
        </form>
        '''

print("HTTP method handlers registered")

## 4. URL Routing and Dynamic Routes

Create dynamic routes that capture parts of the URL as parameters.

In [ ]:
# Example 1: Simple dynamic route with a string parameter
@app.route('/user/<name>')
def greet_user(name):
    """Capture URL parameter - converts to string by default"""
    return f'<h1>Hello, {name}!</h1>'

# Example 2: Using type converters for URL parameters
@app.route('/post/<int:post_id>')
def view_post(post_id):
    """post_id is converted to integer"""
    return f'<h1>Viewing post #{post_id}</h1>'

# Example 3: Multiple URL parameters
@app.route('/user/<username>/post/<int:post_id>')
def user_post(username, post_id):
    """Multiple dynamic parameters"""
    return f'<h1>{username}\'s Post #{post_id}</h1>'

# Example 4: Different type converters
@app.route('/api/<string:resource>/<int:resource_id>')
def api_endpoint(resource, resource_id):
    """Types: string (default), int, float, path, uuid"""
    return f'Resource: {resource}, ID: {resource_id}'

# Example 5: Using path converter for multiple path segments
@app.route('/files/<path:filepath>')
def serve_file(filepath):
    """path converter allows slashes in the parameter"""
    return f'<h1>File path: {filepath}</h1>'

print("Dynamic routes registered")

## 5. Request and Response Objects

Flask provides the `request` object to access incoming request data and allows you to create custom responses.

In [ ]:
# Example 1: Accessing request properties
@app.route('/request-info')
def request_info():
    """Display information about the incoming request"""
    return f'''
    <h2>Request Information</h2>
    <p>Method: {request.method}</p>
    <p>Host: {request.host}</p>
    <p>Remote Address: {request.remote_addr}</p>
    <p>User Agent: {request.user_agent}</p>
    <p>Full Path: {request.full_path}</p>
    '''

# Example 2: Accessing form data
@app.route('/api/submit', methods=['POST'])
def submit_data():
    """Access form data from POST request"""
    username = request.form.get('username', 'Unknown')
    password = request.form.get('password', '')
    return f'Username: {username}, Password length: {len(password)}'

# Example 3: Working with JSON data
@app.route('/api/json', methods=['POST'])
def handle_json():
    """Process JSON data from request"""
    data = request.get_json()  # Parse JSON from request body
    if data:
        name = data.get('name', 'Guest')
        age = data.get('age', 0)
        return jsonify({'message': f'Hello {name}, you are {age} years old'})
    return jsonify({'error': 'No JSON data'}), 400

# Example 4: Accessing request headers
@app.route('/headers')
def show_headers():
    """Display request headers"""
    return f'''
    <h2>Request Headers</h2>
    <p>Content-Type: {request.content_type}</p>
    <p>Accept: {request.headers.get('Accept', 'Not specified')}</p>
    <p>Authorization: {request.headers.get('Authorization', 'None')}</p>
    '''

# Example 5: Custom response with status code and headers
@app.route('/custom-response')
def custom_response():
    """Create a custom response with specific status code and headers"""
    from flask import Response
    response = Response('Custom response text', status=201)
    response.headers['X-Custom-Header'] = 'Custom Value'
    return response

print("Request/Response handlers registered")

## 6. Template Rendering with Jinja2

Use Jinja2 templating to render dynamic HTML pages. Templates are stored in a `templates` folder.

In [ ]:
# Example 1: Basic template rendering with variables
@app.route('/profile/<name>')
def profile(name):
    """Render a template with variable passing"""
    age = 25
    city = 'New York'
    # In real app: return render_template('profile.html', name=name, age=age, city=city)
    # For now, showing the concept with inline HTML
    html = f'<h1>{name}\'s Profile</h1><p>Age: {age}, City: {city}</p>'
    return html

# Example 2: Template with loops and conditionals
@app.route('/products')
def products():
    """Render template with loops"""
    product_list = [
        {'id': 1, 'name': 'Laptop', 'price': 999},
        {'id': 2, 'name': 'Mouse', 'price': 29},
        {'id': 3, 'name': 'Keyboard', 'price': 79},
    ]
    # In real app: return render_template('products.html', products=product_list)
    html = '<h1>Products</h1><ul>'
    for product in product_list:
        html += f'<li>{product["name"]} - ${product["price"]}</li>'
    html += '</ul>'
    return html

# Example 3: Template inheritance concept
# Base template (base.html):
base_template = '''
<!DOCTYPE html>
<html>
<head>
    <title>{% block title %}My App{% endblock %}</title>
</head>
<body>
    <nav>Navigation</nav>
    {% block content %}{% endblock %}
    <footer>Footer</footer>
</body>
</html>
'''

# Child template (page.html):
child_template = '''
{% extends "base.html" %}
{% block title %}Home Page{% endblock %}
{% block content %}
    <h1>Welcome!</h1>
{% endblock %}
'''

print("Template concepts demonstrated")
print("\nJinja2 Template Syntax:")
print("- Variables: {{ variable_name }}")
print("- Conditionals: {% if condition %} ... {% endif %}")
print("- Loops: {% for item in items %} ... {% endfor %}")
print("- Filters: {{ name|upper }}, {{ text|length }}")
print("- Template inheritance: {% extends 'base.html' %}")

## 7. Static Files Management

Serve static files like CSS, JavaScript, and images using the `url_for()` function and a static folder.

In [ ]:
# Static files directory structure:
# project_root/
#   ├── app.py
#   ├── templates/
#   │   └── index.html
#   └── static/
#       ├── css/
#       │   └── style.css
#       ├── js/
#       │   └── script.js
#       └── images/
#           └── logo.png

# Example 1: Using url_for() for static files
@app.route('/static-example')
def static_example():
    """Demonstrate url_for for static files"""
    # In real app, these URLs would be generated dynamically:
    css_url = url_for('static', filename='css/style.css')
    js_url = url_for('static', filename='js/script.js')
    img_url = url_for('static', filename='images/logo.png')
    
    return f'''
    <html>
    <head>
        <link rel="stylesheet" href="{css_url}">
    </head>
    <body>
        <img src="{img_url}" alt="Logo">
        <script src="{js_url}"></script>
    </body>
    </html>
    '''

# Example 2: Example CSS content (static/css/style.css)
css_content = '''
body {
    font-family: Arial, sans-serif;
    margin: 20px;
    background-color: #f5f5f5;
}

h1 {
    color: #333;
}

.container {
    max-width: 800px;
    margin: 0 auto;
    background: white;
    padding: 20px;
    border-radius: 5px;
}
'''

# Example 3: Example JavaScript content (static/js/script.js)
js_content = '''
// Simple JavaScript example
document.addEventListener('DOMContentLoaded', function() {
    console.log('Page loaded');
    
    // Example: Add click handler
    document.addEventListener('click', function(e) {
        if (e.target.tagName === 'BUTTON') {
            console.log('Button clicked:', e.target.textContent);
        }
    });
});
'''

print("Static files structure and url_for() usage demonstrated")
print("\nBest practices for static files:")
print("- Store static files in the 'static' folder")
print("- Always use url_for('static', filename='path') in templates")
print("- This allows Flask to handle caching and file serving efficiently")
print("- Never hardcode paths like '/static/style.css'")

## 8. Error Handling and Status Codes

Handle different error scenarios and return appropriate HTTP status codes.

In [ ]:
# Example 1: Handle 404 Not Found errors
@app.errorhandler(404)
def not_found_error(error):
    """Handle 404 errors"""
    return '<h1>404 - Page Not Found</h1><p>The page you are looking for does not exist.</p>', 404

# Example 2: Handle 500 Internal Server Error
@app.errorhandler(500)
def internal_error(error):
    """Handle 500 errors"""
    return '<h1>500 - Internal Server Error</h1><p>Something went wrong on our end.</p>', 500

# Example 3: Handle custom exceptions
class InvalidAPIUsage(Exception):
    """Custom exception for API usage"""
    def __init__(self, message, status_code=400):
        self.message = message
        self.status_code = status_code

@app.errorhandler(InvalidAPIUsage)
def handle_invalid_usage(error):
    """Handle custom exceptions"""
    response = jsonify({'error': error.message})
    response.status_code = error.status_code
    return response

# Example 4: Return specific status codes from routes
@app.route('/api/users/<int:user_id>')
def get_user(user_id):
    """Return different status codes based on conditions"""
    users = {1: 'Alice', 2: 'Bob'}
    
    if user_id not in users:
        return jsonify({'error': 'User not found'}), 404
    
    return jsonify({'id': user_id, 'name': users[user_id]}), 200

# Example 5: Common HTTP Status Codes
status_codes_reference = {
    '200': 'OK - Request successful',
    '201': 'Created - Resource created successfully',
    '204': 'No Content - Success but no content to return',
    '301': 'Moved Permanently - Resource moved',
    '302': 'Found - Temporary redirect',
    '400': 'Bad Request - Invalid request data',
    '401': 'Unauthorized - Authentication required',
    '403': 'Forbidden - Authenticated but not authorized',
    '404': 'Not Found - Resource does not exist',
    '500': 'Internal Server Error',
    '503': 'Service Unavailable',
}

print("Error handling and status codes registered")
print("\nCommon HTTP Status Codes:")
for code, description in list(status_codes_reference.items())[:7]:
    print(f"  {code}: {description}")

## 9. Working with Query Parameters

Extract and process query string parameters from URLs.

In [ ]:
# Example 1: Basic query parameter access
@app.route('/search')
def search():
    """Access query parameters from URL"""
    # URL: /search?q=python&limit=10
    query = request.args.get('q', 'default search')
    limit = request.args.get('limit', 5, type=int)
    
    return f'<h1>Search Results</h1><p>Query: {query}, Limit: {limit}</p>'

# Example 2: Multiple query parameters
@app.route('/filter')
def filter_data():
    """Handle multiple query parameters"""
    # URL: /filter?category=books&price_min=10&price_max=50&sort=price
    category = request.args.get('category', 'all')
    price_min = request.args.get('price_min', 0, type=float)
    price_max = request.args.get('price_max', 1000, type=float)
    sort = request.args.get('sort', 'name')
    
    return f'''
    <h2>Filtered Results</h2>
    <p>Category: {category}</p>
    <p>Price Range: ${price_min} - ${price_max}</p>
    <p>Sorted by: {sort}</p>
    '''

# Example 3: Check if query parameter exists
@app.route('/pagination')
def pagination():
    """Implement pagination using query parameters"""
    page = request.args.get('page', 1, type=int)
    per_page = request.args.get('per_page', 10, type=int)
    
    # Validate parameters
    if page < 1:
        return jsonify({'error': 'Page must be >= 1'}), 400
    if per_page > 100:
        return jsonify({'error': 'per_page cannot exceed 100'}), 400
    
    # Calculate offset
    offset = (page - 1) * per_page
    
    return jsonify({
        'page': page,
        'per_page': per_page,
        'offset': offset,
        'items': [f'Item {i}' for i in range(offset, offset + per_page)]
    })

# Example 4: Get all query parameters
@app.route('/debug-params')
def debug_params():
    """Display all query parameters"""
    all_params = request.args.to_dict()
    
    html = '<h2>Query Parameters</h2><ul>'
    for key, value in all_params.items():
        html += f'<li>{key}: {value}</li>'
    html += '</ul>'
    
    return html

# Example 5: URL construction with query parameters
@app.route('/construct-urls')
def construct_urls():
    """Show how to construct URLs with query parameters"""
    # Using url_for with query parameters
    search_url = url_for('search', q='python', limit=20)
    filter_url = url_for('filter_data', category='books', sort='price')
    
    return f'''
    <h2>Generated URLs</h2>
    <p>Search URL: {search_url}</p>
    <p>Filter URL: {filter_url}</p>
    '''

print("Query parameter handling demonstrated")
print("\nQuery Parameter Examples:")
print("  /search?q=python&limit=10")
print("  /filter?category=books&price_min=10&price_max=50")
print("  /pagination?page=2&per_page=20")

## 10. Running the Flask Development Server

Start and configure the Flask development server for local testing.

In [ ]:
# Example 1: Basic server startup
# To run this in real life, save code to app.py and run:
# python app.py
# OR
# flask run

# if __name__ == '__main__':
#     app.run(debug=True)

# Example 2: Server configuration options
print("Flask Server Configuration Options:")
print("\nBasic startup with app.run():")
print("  app.run()")
print("  - Default: http://localhost:5000")
print("")

print("Common configurations:")
print("  app.run(debug=True)           # Enable debug mode")
print("  app.run(host='0.0.0.0')       # Listen on all interfaces")
print("  app.run(port=8000)            # Custom port")
print("  app.run(debug=True, port=5001) # Debug + custom port")

# Example 3: Different environments
configurations = {
    'Development': {
        'debug': True,
        'host': 'localhost',
        'port': 5000,
        'use_reloader': True,
        'description': 'Use for local development'
    },
    'Testing': {
        'debug': False,
        'testing': True,
        'description': 'Use for running tests'
    },
    'Production': {
        'debug': False,
        'host': '0.0.0.0',
        'port': 80,
        'description': 'Use with production servers like Gunicorn'
    }
}

print("\n\nEnvironment Configurations:")
for env, config in configurations.items():
    print(f"\n{env}:")
    for key, value in config.items():
        if key != 'description':
            print(f"  {key}: {value}")
    print(f"  Note: {config['description']}")

# Example 4: Running with Gunicorn (production)
print("\n\nProduction Deployment:")
print("  # Install Gunicorn:")
print("  pip install gunicorn")
print("")
print("  # Run with Gunicorn:")
print("  gunicorn -w 4 -b 0.0.0.0:8000 app:app")
print("  (-w: number of workers, -b: bind address)")

# Example 5: Debug mode explanation
print("\n\nDebug Mode Benefits:")
print("  ✓ Automatic code reloading on file changes")
print("  ✓ Interactive debugger on errors")
print("  ✓ Better error messages")
print("  ✓ Template auto-reloading")
print("\n  WARNING: Never use debug=True in production!")

# Example 6: Testing the application
print("\n\nTesting your Flask Application:")
print("  # Create a test client:")
print("  with app.test_client() as client:")
print("      response = client.get('/')")
print("      assert response.status_code == 200")

print("\n\nServer started successfully!")

## Bonus: Best Practices and Tips

Key recommendations for developing Flask applications.

In [ ]:
# Best Practices for Flask Development

best_practices = """
1. PROJECT STRUCTURE
   ├── app.py (or app/__init__.py for larger apps)
   ├── config.py (configuration settings)
   ├── requirements.txt (dependencies)
   ├── templates/ (HTML files)
   │   ├── base.html
   │   └── pages/
   ├── static/ (CSS, JS, images)
   │   ├── css/
   │   ├── js/
   │   └── images/
   └── tests/ (test files)

2. CONFIGURATION MANAGEMENT
   - Use config.py for settings
   - Don't hardcode credentials
   - Use environment variables
   - Separate dev/test/prod configs

3. ERROR HANDLING
   - Always handle 404 and 500 errors
   - Validate user input
   - Use try-except for database operations
   - Log errors for debugging

4. SECURITY
   - Use app.secret_key for sessions
   - Validate and sanitize all inputs
   - Use HTTPS in production
   - Never expose sensitive info in error messages
   - Implement CSRF protection for forms

5. PERFORMANCE
   - Cache static files properly
   - Use blueprints for modular code
   - Minimize database queries
   - Consider using a production server (Gunicorn)

6. CODE ORGANIZATION
   - Use blueprints for feature separation
   - Create helper functions for common tasks
   - Use decorators for repeated logic
   - Keep routes simple and focused

7. TESTING
   - Write unit tests for routes
   - Test error scenarios
   - Use pytest for better testing
   - Aim for good code coverage

8. DOCUMENTATION
   - Document your API endpoints
   - Add docstrings to functions
   - Create a README.md
   - Document configuration options

9. DEPENDENCIES
   - Keep requirements.txt updated
   - Use virtual environments
   - Minimize external dependencies
   - Check for security updates

10. DEPLOYMENT
    - Use environment variables for secrets
    - Enable HTTPS
    - Use a production WSGI server
    - Set up proper logging
    - Monitor application performance
"""

print(best_practices)

# Example: Project structure code
project_example = {
    'app.py': 'Main application file',
    'config.py': 'Configuration settings',
    'requirements.txt': 'pip freeze > requirements.txt',
    'templates/': 'HTML templates',
    'static/': 'CSS, JS, images',
    'tests/': 'Unit tests'
}

# Example: Secure configuration
print("\n\nExample Configuration Management:")
print("""
# config.py
import os

class Config:
    DEBUG = False
    SECRET_KEY = os.environ.get('SECRET_KEY', 'dev-key-change-this')
    DATABASE_URL = os.environ.get('DATABASE_URL')

class DevelopmentConfig(Config):
    DEBUG = True

class ProductionConfig(Config):
    DEBUG = False

# app.py
from flask import Flask
from config import DevelopmentConfig

app = Flask(__name__)
app.config.from_object(DevelopmentConfig)
""")